In [2]:
import pandas as pd 
import numpy as np
import requests

In [2]:
df_2020 = pd.read_csv("../data/2020_US_County_Level_Presidential_Results.csv")
df_2016 = pd.read_csv("../data/2016_US_County_Level_Presidential_Results.csv")

#calculate the winner for each county in 2020 and 2016
df_2020["win_2020"] = np.where(df_2020["per_gop"] > df_2020["per_dem"], "gop", "dem")
df_2016["win_2016"] = np.where(df_2016["per_gop"] > df_2016["per_dem"], "gop", "dem")

In [3]:
df_2016["combined_fips"] = df_2016["combined_fips"].astype(str).str.zfill(5)

In [4]:
df_2016["combined_fips"] = pd.to_numeric(df_2016["combined_fips"])

In [5]:
df = df_2016[["combined_fips", "win_2016"]].merge(
    df_2020[["county_fips", "win_2020"]],
    left_on="combined_fips", right_on="county_fips", how="inner")

df.head()

,combined_fips,win_2016,county_fips,win_2020
0,1001,gop,1001,gop
1,1003,gop,1003,gop
2,1005,gop,1005,gop
3,1007,gop,1007,gop
4,1009,gop,1009,gop


In [6]:
df["flip"] = np.where(df["win_2016"] != df["win_2020"], 1, 0)
df.head()

,combined_fips,win_2016,county_fips,win_2020,flip
0,1001,gop,1001,gop,0
1,1003,gop,1003,gop,0
2,1005,gop,1005,gop,0
3,1007,gop,1007,gop,0
4,1009,gop,1009,gop,0


In [7]:
df.head()

,combined_fips,win_2016,county_fips,win_2020,flip
0,1001,gop,1001,gop,0
1,1003,gop,1003,gop,0
2,1005,gop,1005,gop,0
3,1007,gop,1007,gop,0
4,1009,gop,1009,gop,0


In [8]:
cols = ['combined_fips']  

summary = pd.DataFrame({
    'Mean': df[cols].mean(),
    'Median': df[cols].median(),
    'Std Dev': df[cols].std(),
    'Variance': df[cols].var(),
    'Min': df[cols].min(),
    'Max': df[cols].max(),
    'Range': df[cols].max() - df[cols].min(),
    'IQR': df[cols].quantile(0.75) - df[cols].quantile(0.25),
    'Skew': df[cols].skew(),
    'Null %': df[cols].isna().mean() * 100
})

print(summary)

                       Mean   Median      Std Dev      Variance   Min    Max  \
combined_fips  30646.730633  29207.0  14984.49836  2.245352e+08  1001  56045   

               Range      IQR      Skew  Null %  
combined_fips  55044  26966.0 -0.073672     0.0  


In [ ]:
API_KEY = " "

def get_data(year):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": "NAME,B01001_001E, B19013_001E",
        "for": "county:*",
        "in": "state:*",
        "key": API_KEY
    }
    r = requests.get(url, params=params)
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    
    # create FIPS
    df["fips"] = df["state"] + df["county"]
    df["year"] = year
    
    return df

df_2016 = get_data(2016)
df_2020 = get_data(2020)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# B02001_002E - white
# B02001_003E - black
# B19013_001E - med household income
# B17001_002E - poverty 
# B01001_* - sex by age
# B23025_004E - employed
# B15003_* - educational attainment
# B25003_* - homeownership
# B05002_* - immigration 

In [ ]:
API_KEY = "616c188b57a3ed81801350dcdfb3be29713e93fa"

#function to get all variables for a given table prefix
def get_table_vars(year, table_prefix):
    meta_url = f"https://api.census.gov/data/{year}/acs/acs5/variables.json"
    res = requests.get(meta_url).json()
    
    vars_list = [
        var for var in res["variables"].keys()
        if var.startswith(table_prefix) and var.endswith("E")
    ]
    
    return vars_list

def build_variable_list(year):
    base_vars = [
        "B02001_002E",  #white
        "B02001_003E",  #black
        "B19013_001E",  #med hh income
        "B17001_002E",  #poverty
        "B23025_004E",   #employed
        "B01001_001E" #total population
    ]
    
    #tables that have multiple 
    sex_age = get_table_vars(year, "B01001")
    education = get_table_vars(year, "B15003")
    homeownership = get_table_vars(year, "B25003")
    immigration = get_table_vars(year, "B05002")
    
    all_vars = base_vars + sex_age + education + homeownership + immigration
    
    return ",".join(all_vars)

def chunk_list(lst, size=45): #to not hit api limit 
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

def get_data(year):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    
    vars_list = build_variable_list(year).split(",")
    
    dfs = []

    for i, chunk in enumerate(chunk_list(vars_list)):
        if i == 0:
            get_vars = ["NAME"] + chunk
        else:
            get_vars = chunk

        params = {
            "get": ",".join(get_vars),
            "for": "county:*",
            "in": "state:*",
            "key": API_KEY
        }
        
        r = requests.get(url, params=params)
        data = r.json()
        df_chunk = pd.DataFrame(data[1:], columns=data[0])
        
        dfs.append(df_chunk)
    
    #merge on fips codes 
    df = dfs[0]
    for d in dfs[1:]:
        df = df.merge(d, on=["state", "county"])
    
    df["fips"] = df["state"] + df["county"]
    df["year"] = year

    working_cols = [ #all ages 
    "B01001_007E","B01001_008E","B01001_009E",  
    "B01001_010E","B01001_011E","B01001_012E",
    "B01001_031E","B01001_032E","B01001_033E",  
    "B01001_034E","B01001_035E","B01001_036E"
    ]
    df["working_age_pop"] = df[working_cols].astype(float).sum(axis=1)

    college_cols = ["B15003_022E","B15003_023E","B15003_024E","B15003_025E"]
    df["college_pop"] = df[college_cols].astype(float).sum(axis=1)

    df["home_total"] = df["B25003_001E"].astype(float)
    df["homeowner"] = df["B25003_002E"].astype(float) 
    df["renter"] = df["B25003_003E"].astype(float)

    df["imm_total"] = df["B05002_001E"].astype(float)
    df["foreign_born"] = df["B05002_013E"].astype(float)

    df = df.rename(columns={"B02001_002E": "White", "B02001_003E": "Black", "B19013_001E": "Med_HH_Income", "B17001_002E": "Poverty", "B23025_004E": "Employed"})
    df = df.rename(columns={"B01001_001E": "Total_Population"})
    
    df.drop(columns=[col for col in df.columns if col.startswith("B")], inplace=True)
    df.drop(columns=['NAME'], inplace=True)

    return df

df20 = get_data(2020)
df16 = get_data(2016)

In [22]:
df = df16.merge(df20, on="fips", suffixes=("_2016", "_2020"))

In [23]:
pct_change_df = pd.DataFrame()
pct_change_df["fips"] = df["fips"]

for col in df.columns:
    if col.endswith("_2016"):
        base_col = col.replace("_2016", "")
        col_2020 = base_col + "_2020"
        
        if col_2020 in df.columns:
            pct_change_df[base_col + "_pct_change"] = (
                (df[col_2020].astype(float) - df[col].astype(float)) 
                / df[col].astype(float) 
            ) *100

pct_change_df.head()

,fips,White_pct_change,Med_HH_Income_pct_change,Poverty_pct_change,Employed_pct_change,state_pct_change,county_pct_change,year_pct_change,working_age_pop_pct_change,college_pop_pct_change,home_total_pct_change,homeowner_pct_change,renter_pct_change,imm_total_pct_change,foreign_born_pct_change
0,29229,-2.441082,20.045126,-16.710526,-0.940018,0.0,0.0,0.198413,-0.895062,-3.165468,-5.652468,5.298805,-28.800000,-0.663837,-34.463277
1,29047,2.819864,13.100060,-4.715580,7.760220,0.0,0.0,0.198413,6.304789,10.407404,4.666870,2.414379,9.889615,5.724151,24.341854
2,29065,-2.798880,12.346134,-19.293637,9.896008,0.0,0.0,0.198413,-6.268222,19.897959,7.913058,8.059914,7.546049,-0.385159,79.166667
3,29195,-2.674494,17.074671,-11.979723,-1.138272,0.0,0.0,0.198413,2.602839,17.993686,-6.260672,-1.520527,-16.050244,-1.214784,-1.939292
4,29227,-2.285147,5.616578,18.702290,-8.562691,0.0,0.0,0.198413,-2.678571,1.153846,-11.886587,-2.802360,-37.656904,-2.911208,33.333333


In [ ]:
#calculate uncertainty statitics 
df = pct_change_df

cols = ['White_pct_change', 'Med_HH_Income_pct_change', 'Poverty_pct_change', 'Employed_pct_change',
        'working_age_pop_pct_change', 'college_pop_pct_change', 'homeowner_pct_change', 'renter_pct_change',
         'imm_total_pct_change', 'foreign_born_pct_change']  

summary = pd.DataFrame({
    'Mean': df[cols].mean(),
    'Median': df[cols].median(),
    'Std Dev': df[cols].std(),
    'Variance': df[cols].var(),
    'Min': df[cols].min(),
    'Max': df[cols].max(),
    'Range': df[cols].max() - df[cols].min(),
    'IQR': df[cols].quantile(0.75) - df[cols].quantile(0.25),
    'Skew': df[cols].skew(),
    'Null %': df[cols].isna().mean() * 100
})

print(summary)

                                  Mean     Median       Std Dev      Variance  \
White_pct_change             -2.146231  -1.910017      7.242812  5.245832e+01   
Med_HH_Income_pct_change   -397.900161  14.929003  23421.091051  5.485475e+08   
Poverty_pct_change           -9.015561 -11.563868     29.083052  8.458239e+02   
Employed_pct_change           2.533048   2.077243     11.656764  1.358801e+02   
working_age_pop_pct_change   -0.260662  -0.462725     11.730516  1.376050e+02   
college_pop_pct_change       11.871153  11.225036     18.938118  3.586523e+02   
homeowner_pct_change          2.972185   2.886137      7.560402  5.715969e+01   
renter_pct_change             0.079564  -0.281294     17.785167  3.163122e+02   
imm_total_pct_change          0.285176  -0.353096      8.171311  6.677032e+01   
foreign_born_pct_change            inf   4.403682           NaN           NaN   

                                     Min          Max         Range  \
White_pct_change           -8.679225e

/home/nylup/DS4320/ds4320-project2/env/lib/python3.10/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/nylup/DS4320/ds4320-project2/env/lib/python3.10/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/home/nylup/DS4320/ds4320-project2/env/lib/python3.10/site-packages/pandas/core/nanops.py:1256: RuntimeWarning: invalid value encountered in subtract
  adjusted = values - mean


In [3]:
#read in created csvs for election results and census data
election = pd.read_csv("../data/county_results.csv")
census = pd.read_csv("../data/census_data.csv")
df = election.merge(census, on="fips")
df.head()

,fips,win_2016,county_fips,win_2020,flip,White_pct_change,Med_HH_Income_pct_change,Poverty_pct_change,Employed_pct_change,working_age_pop_pct_change,college_pop_pct_change,home_total_pct_change,homeowner_pct_change,renter_pct_change,imm_total_pct_change,foreign_born_pct_change
0,1001,gop,1001,gop,0,-0.003805,0.091960,0.253397,0.013107,0.022432,0.195473,0.036490,0.057169,-0.019885,0.010718,0.287549
1,1003,gop,1003,gop,0,0.081553,0.202297,-0.227467,0.125523,0.057320,0.200387,0.118405,0.200557,-0.090049,0.094126,0.145025
2,1005,gop,1005,gop,0,-0.067820,0.030451,0.012350,-0.031803,-0.080553,-0.126374,0.021925,-0.009264,0.077133,-0.059668,-0.160315
3,1007,gop,1007,gop,0,-0.013356,0.300307,0.107670,-0.006105,-0.025844,-0.040212,0.029938,0.060363,-0.050804,-0.008772,0.160656
4,1009,gop,1009,gop,0,-0.014562,0.058643,-0.168838,0.057565,0.012828,0.024267,0.028420,-0.007506,0.162199,0.000884,0.019878
